In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
micro = pd.read_csv(f'{RAW_DIR}/Datamart/AllMGB_micro_20150525-20250201.csv')
enc = pd.read_csv(f'{RAW_DIR}/edw/encounters/MGB_encounters_20240715.csv')
mrn_map = pd.read_csv(f'{RAW_DIR}/edw/MRN_map/EDW_MRN_map_20240713.csv')
demo = pd.read_csv(f'{RAW_DIR}/edw/demographics/MGB_demographics_20240715.csv')

In [ ]:
enc = enc.rename(columns={'PatientID':'patientID',
                          'PatientEncounterID':'patientEncounterID',
                          'visit_type':'adtPatientClassification'})

demo['patientID'] = demo['PatientID']
demo['birthDT'] = demo['DOB']

enc = enc.merge(demo[['patientID','birthDT']],
                on = ['patientID'],
                how = 'inner')
enc

In [ ]:
mrn_map = mrn_map.rename(columns={'PatientID':'patientID',
                                  'MRN':'mrn'})
mrn_map

## Create or replace the corhort table named microbiology_cultures_cohort

In [ ]:
micro_cultures = micro[['hospital','MRN','coll_datetime','accession',
                        'site_clean','org_clean','drug','pheno_merged']]

## Filter for only Urine, Blood and Respiratory cultures
micro_cultures = micro_cultures[(micro_cultures['site_clean'] == 'URINE') |
                                (micro_cultures['site_clean'] == 'RESPIRATORY_TRACT') |
                                (micro_cultures['site_clean'] == 'BLOOD')]

## Map MRN to patientID
micro_cultures = micro_cultures.merge(mrn_map[['patientID','hospital','mrn']],
                                      left_on = ['hospital','MRN'],
                                      right_on = ['hospital','mrn'])

## Prefix accession with hospital to ensure uniqueness across hospitals
## (different hospitals may independently assign the same accession number)
micro_cultures['accession'] = micro_cultures['hospital'] + '_' + micro_cultures['accession'].astype(str)

## Get patientEncounterID via range join: culture collection time must fall
## within the encounter's admission–discharge window.
## Bug fix: old code merged on admit_date (date only). A patient with two
## encounters starting on the same calendar date would produce duplicate rows
## for every culture order, causing same-patient / different-encounter conflicts.
micro_cultures['coll_datetime'] = pd.to_datetime(micro_cultures['coll_datetime'])
enc['hospitalAdmitDTS'] = pd.to_datetime(enc['hospitalAdmitDTS'])
enc['hospitalDischargeDTS'] = pd.to_datetime(enc['hospitalDischargeDTS'])

## Merge all encounters per patient, then keep only the one the culture falls in
micro_cultures = micro_cultures.merge(
    enc[['patientID','patientEncounterID','adtPatientClassification',
         'hospitalAdmitDTS','hospitalDischargeDTS','birthDT']],
    on='patientID',
    how='inner'
)

## For active encounters with no discharge recorded, treat window as open-ended
micro_cultures = micro_cultures[
    (micro_cultures['coll_datetime'] >= micro_cultures['hospitalAdmitDTS']) &
    (micro_cultures['coll_datetime'] <= micro_cultures['hospitalDischargeDTS'].fillna(pd.Timestamp('2099-12-31')))
].copy()

## Filter for adult patients only (age at admission)
micro_cultures['age'] = (micro_cultures['hospitalAdmitDTS'] - pd.to_datetime(micro_cultures['birthDT'])).dt.days / 365.25
micro_cultures = micro_cultures[micro_cultures['age'] > 18]

# Sort by patientID and datetime — required for shift to work correctly
micro_cultures = micro_cultures.sort_values(['patientID', 'coll_datetime'])

micro_cultures

In [ ]:
# Get Previous Accession
accessions = micro_cultures[['patientID','hospital','mrn','accession','site_clean','coll_datetime']].copy()
accessions = accessions.drop_duplicates()

# Shift coll_datetime within each patient to get the previous collection date
accessions['prev_coll'] = accessions.groupby('patientID')['coll_datetime'].shift(1)

# Compute the difference in days
accessions['days_since_prev'] = (accessions['coll_datetime'] - accessions['prev_coll']).dt.days

# Get Accession with no test within 14 days
accessions = accessions[(accessions['days_since_prev'].isna()) | (accessions['days_since_prev'] > 14)]

accessions

In [ ]:
def categorize_susceptibility(value):
    if pd.isna(value):  # Keep NULL (NaN) values unchanged
        return None
    susceptibility_map = {
        'Susceptible': 'Susceptible',
        'Resistant': 'Resistant',
        'Intermediate': 'Intermediate',
        'Negative': 'Susceptible',
        'Positive': 'Resistant',
        'Susceptible dose dependent': 'Intermediate',
        'Non-susceptible': 'Resistant',
        'Susceptible dose-dependent': 'Intermediate',
    }
    return susceptibility_map.get(value, 'Unknown')  # Default to 'Unknown' for unexpected values


In [ ]:
## Feature Engineering and Cleaning
# Bug fix: merge on (patientID, accession) instead of isin(accession).
# The old isin filter only checked the accession value, not which patient it
# belonged to. If accession "X" was removed for Patient B by the 14-day rule
# but kept for Patient A, isin would incorrectly re-include Patient B's rows.
micro_removed_prior_cultures = micro_cultures.merge(
    accessions[['patientID', 'accession']].drop_duplicates(),
    on=['patientID', 'accession']
)
micro_removed_prior_cultures['susceptibility'] = micro_removed_prior_cultures['pheno_merged'].apply(categorize_susceptibility)
micro_removed_prior_cultures['was_positive'] = np.where(micro_removed_prior_cultures['org_clean'].isna(),0,1)
micro_removed_prior_cultures = micro_removed_prior_cultures[['patientID','patientEncounterID','accession','coll_datetime','adtPatientClassification','site_clean','was_positive','org_clean','drug','susceptibility']]

## Feature Rename
micro_removed_prior_cultures = micro_removed_prior_cultures.rename(columns = {'patientID':'anon_id',
                                                                              'patientEncounterID':'pat_enc_csn_id_coded',
                                                                              'accession':'order_proc_id_coded',
                                                                              'coll_datetime':'order_time_jittered_utc',
                                                                              'adtPatientClassification':'ordering_mode',
                                                                              'site_clean':'culture_description',
                                                                              'org_clean':'organism',
                                                                              'drug':'antibiotic',})

micro_removed_prior_cultures

In [ ]:
micro_removed_prior_cultures.to_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv',
                                    index = False)

## Deidentification

In [ ]:
# Deliberately unseeded: a fixed seed in published code would let anyone
# regenerate the ID maps and date offsets from the number of unique IDs.
rng = np.random.default_rng()

# Function to generate random mapping
def random_id_map(series):
    unique_vals = series.unique()
    random_ids = rng.permutation(range(1, len(unique_vals) + 1))
    return dict(zip(unique_vals, random_ids))

# Create mappings
anon_id_map = random_id_map(micro_removed_prior_cultures['anon_id'])
enc_id_map = random_id_map(micro_removed_prior_cultures['pat_enc_csn_id_coded'])
order_proc_id_map = random_id_map(micro_removed_prior_cultures['order_proc_id_coded'])

# Convert mapping to DataFrame and save
pd.DataFrame(anon_id_map.items(), columns=['anon_id', 'anon_id_deid']).to_csv(f'{DEID_DIR}/anon_id_map.csv', index=False)
pd.DataFrame(enc_id_map.items(), columns=['pat_enc_csn_id_coded', 'pat_enc_csn_id_deid']).to_csv(f'{DEID_DIR}/enc_id_map.csv', index=False)
pd.DataFrame(order_proc_id_map.items(), columns=['order_proc_id_coded', 'order_proc_id_deid']).to_csv(f'{DEID_DIR}/order_proc_id_map.csv', index=False)

# Apply mappings
deid_micro_removed_prior_cultures = micro_removed_prior_cultures.copy()
deid_micro_removed_prior_cultures['anon_id'] = deid_micro_removed_prior_cultures['anon_id'].map(anon_id_map)
deid_micro_removed_prior_cultures['pat_enc_csn_id_coded'] = deid_micro_removed_prior_cultures['pat_enc_csn_id_coded'].map(enc_id_map)
deid_micro_removed_prior_cultures['order_proc_id_coded'] = deid_micro_removed_prior_cultures['order_proc_id_coded'].map(order_proc_id_map)


deid_micro_removed_prior_cultures


In [ ]:
deid_micro_removed_prior_cultures.to_csv(f'{DEID_DIR}/microbiology_cultures_cohort.csv',
                                         index = False)

In [ ]:
micro_removed_prior_cultures[['anon_id','ordering_mode']].drop_duplicates().groupby(['ordering_mode']).count()

In [ ]:
# Shift all dates per patient by a consistent random offset
def shift_dates_per_patient(df, patient_id_col):
    patient_offsets = {
        pid: pd.to_timedelta(rng.integers(int(-84 * 365.25), int(84 * 365.25)), unit='D')
        for pid in df[patient_id_col].unique()
    }
    return patient_offsets

In [ ]:
shift_dates_dict = shift_dates_per_patient(deid_micro_removed_prior_cultures,
                                           'anon_id')

In [ ]:
df = pd.DataFrame([
    {"anon_id": int(k), "date_offset": str(v)}
    for k, v in shift_dates_dict.items()
])

df.to_csv(f"{DEID_DIR}/date_shift_map.csv", index=False)
